# Foundations of RAG: Vanilla RAG, Embeddings, Chunking, BM25,Hybrid


##  RAG

### Indexing
```text
Documents → Chunking → Representations → Index
```
### Querying
```text
Question → Retrieval → Top-k context → LLM → Answer
```

In [1]:
# Install dependencies if needed.
# Run this cell once in a fresh environment.

%pip install -q sentence-transformers rank-bm25 scikit-learn transformers accelerate

In [2]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

np.random.seed(42)

print("Environment ready.")

Environment ready.


# Create a small knowledge base

In [3]:
documents = [
    {
        "id": "doc_1",
        "title": "Python",
        "text": (
            "Python is a high-level programming language known for readable syntax. "
            "It was created by Guido van Rossum and first released in 1991. "
            "Python is widely used for automation, data science, web development, "
            "scientific computing, and machine learning."
        ),
    },
    {
        "id": "doc_2",
        "title": "Supervised Learning",
        "text": (
            "Supervised learning trains a machine learning model using labeled examples. "
            "The model learns a relationship between input features and known target labels. "
            "Classification and regression are common supervised learning tasks."
        ),
    },
    {
        "id": "doc_3",
        "title": "Unsupervised Learning",
        "text": (
            "Unsupervised learning works with data that does not have target labels. "
            "The algorithm attempts to discover structure or patterns in the data. "
            "Clustering and dimensionality reduction are common unsupervised learning methods."
        ),
    },
    {
        "id": "doc_4",
        "title": "RAG",
        "text": (
            "Retrieval-Augmented Generation, or RAG, combines information retrieval with "
            "language generation. A retriever finds relevant passages from an external "
            "knowledge base and an LLM uses those passages as context to generate an answer."
        ),
    },
    {
        "id": "doc_5",
        "title": "BM25",
        "text": (
            "BM25 is a lexical information retrieval algorithm. It scores documents using "
            "term frequency, inverse document frequency, and document length normalization. "
            "It is especially useful when exact words, identifiers, product names, or error codes matter."
        ),
    },
    {
        "id": "doc_6",
        "title": "Embeddings",
        "text": (
            "An embedding model converts text into a numerical vector. Dense vector retrieval "
            "uses similarity between query and document vectors to find passages with related meaning. "
            "Cosine similarity is a common similarity measure."
        ),
    },
    {
        "id": "doc_7",
        "title": "SPLADE",
        "text": (
            "SPLADE is a neural sparse retrieval approach. It produces sparse representations "
            "over vocabulary terms and can learn lexical expansion, allowing a query to retrieve "
            "documents that use related or expanded terminology."
        ),
    },
    {
        "id": "doc_8",
        "title": "ColBERT",
        "text": (
            "ColBERT is a late-interaction retrieval architecture. Instead of representing a "
            "document with only one vector, it represents tokens with multiple vectors and compares "
            "query token representations with document token representations."
        ),
    },
]

df = pd.DataFrame(documents)
df

,id,title,text
0,doc_1,Python,Python is a high-level programming language kn...
1,doc_2,Supervised Learning,Supervised learning trains a machine learning ...
2,doc_3,Unsupervised Learning,Unsupervised learning works with data that doe...
3,doc_4,RAG,"Retrieval-Augmented Generation, or RAG, combin..."
4,doc_5,BM25,BM25 is a lexical information retrieval algori...
5,doc_6,Embeddings,An embedding model converts text into a numeri...
6,doc_7,SPLADE,SPLADE is a neural sparse retrieval approach. ...
7,doc_8,ColBERT,ColBERT is a late-interaction retrieval archit...


# Chunking


In [ ]:
def fixed_size_chunks(text, chunk_size=40, overlap=10):
    words = text.split()
    chunks = []

    step = max(1, chunk_size - overlap)
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break

    return chunks

 # Chunking text into sentences level.
def sentence_chunks(text):
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


example_text = documents[3]["text"]

print("Fixed-size chunks:")
for i, chunk in enumerate(fixed_size_chunks(example_text, chunk_size=12, overlap=3)):
    print(f"{i}: {chunk}")

print("\nSentence chunks:")
for i, chunk in enumerate(sentence_chunks(example_text)):
    print(f"{i}: {chunk}")

Fixed-size chunks:
0: Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation. A retriever
1: generation. A retriever finds relevant passages from an external knowledge base and
2: knowledge base and an LLM uses those passages as context to generate
3: context to generate an answer.

Sentence chunks:
0: Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation.
1: A retriever finds relevant passages from an external knowledge base and an LLM uses those passages as context to generate an answer.


In [5]:
# Build a chunk-level corpus for the rest of the notebook.
chunks = []

for doc in documents:
    for idx, chunk in enumerate(fixed_size_chunks(doc["text"], chunk_size=35, overlap=8)):
        chunks.append({
            "chunk_id": f'{doc["id"]}_chunk_{idx}',
            "doc_id": doc["id"],
            "title": doc["title"],
            "text": chunk,
        })

chunks_df = pd.DataFrame(chunks)
print("Number of chunks:", len(chunks_df))
chunks_df.head()

Number of chunks: 9


,chunk_id,doc_id,title,text
0,doc_1_chunk_0,doc_1,Python,Python is a high-level programming language kn...
1,doc_1_chunk_1,doc_1,Python,"automation, data science, web development, sci..."
2,doc_2_chunk_0,doc_2,Supervised Learning,Supervised learning trains a machine learning ...
3,doc_3_chunk_0,doc_3,Unsupervised Learning,Unsupervised learning works with data that doe...
4,doc_4_chunk_0,doc_4,RAG,"Retrieval-Augmented Generation, or RAG, combin..."


#  Dense embeddings


In [6]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

document_embeddings = embedder.encode(
    chunks_df["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)

document_embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(9, 384)

## Query embedding and cosine similarity

In [7]:
def dense_search(query, top_k=5):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
    )

    scores = np.dot(document_embeddings, query_embedding[0])

    result = chunks_df.copy()
    result["score"] = scores

    return result.sort_values("score", ascending=False).head(top_k)


query = "Who created Python?"
dense_results = dense_search(query, top_k=5)

dense_results[["title", "text", "score"]]

,title,text,score
0,Python,Python is a high-level programming language kn...,0.709822
1,Python,"automation, data science, web development, sci...",0.275465
8,ColBERT,ColBERT is a late-interaction retrieval archit...,0.126674
4,RAG,"Retrieval-Augmented Generation, or RAG, combin...",0.101291
5,BM25,BM25 is a lexical information retrieval algori...,0.100393


# Vanilla RAG retrieval


In [8]:
def format_context(results):
    return "\n\n".join(
        f"[{row.title}] {row.text}"
        for row in results.itertuples()
    )

context = format_context(dense_results.head(3))

print(context)

[Python] Python is a high-level programming language known for readable syntax. It was created by Guido van Rossum and first released in 1991. Python is widely used for automation, data science, web development, scientific computing, and

[Python] automation, data science, web development, scientific computing, and machine learning.

[ColBERT] ColBERT is a late-interaction retrieval architecture. Instead of representing a document with only one vector, it represents tokens with multiple vectors and compares query token representations with document token representations.


#  BM25

It focuses on words and their importance rather than representing the entire passage as one dense semantic vector.


In [9]:
tokenized_corpus = [
    chunk.lower().split()
    for chunk in chunks_df["text"]
]

bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=5):
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)

    result = chunks_df.copy()
    result["score"] = scores

    return result.sort_values("score", ascending=False).head(top_k)


bm25_results = bm25_search("Python programming language", top_k=5)

bm25_results[["title", "text", "score"]]

,title,text,score
0,Python,Python is a high-level programming language kn...,4.947795
4,RAG,"Retrieval-Augmented Generation, or RAG, combin...",1.057317
1,Python,"automation, data science, web development, sci...",0.000000
2,Supervised Learning,Supervised learning trains a machine learning ...,0.000000
3,Unsupervised Learning,Unsupervised learning works with data that doe...,0.000000


# Compare dense retrieval and BM25


In [10]:
test_queries = [
    "Who made Python?",
    "What is retrieval augmented generation?",
    "What is ERR_CONNECTION_RESET?",
    "How do models learn from labeled examples?",
]

for q in test_queries:
    print("=" * 80)
    print("QUERY:", q)

    print("\nDense:")
    print(dense_search(q, 2)[["title", "score"]].to_string(index=False))

    print("\nBM25:")
    print(bm25_search(q, 2)[["title", "score"]].to_string(index=False))

QUERY: Who made Python?

Dense:
 title    score
Python 0.673334
Python 0.269890

BM25:
 title  score
Python    0.0
Python    0.0
QUERY: What is retrieval augmented generation?

Dense:
 title    score
   RAG 0.690822
SPLADE 0.388256

BM25:
  title    score
   BM25 0.857172
ColBERT 0.740443
QUERY: What is ERR_CONNECTION_RESET?

Dense:
 title    score
Python 0.126599
  BM25 0.104516

BM25:
 title   score
  BM25 0.50811
Python 0.50288
QUERY: How do models learn from labeled examples?

Dense:
                title    score
  Supervised Learning 0.656371
Unsupervised Learning 0.385955

BM25:
              title    score
Supervised Learning 1.719997
             SPLADE 1.694321


#  Hybrid retrieval

```text
hybrid_score =
    alpha * dense_score
    +
    (1 - alpha) * bm25_score
```


In [11]:
def minmax_normalize(values):
    values = np.asarray(values, dtype=float)
    low, high = values.min(), values.max()

    if high - low < 1e-12:
        return np.zeros_like(values)

    return (values - low) / (high - low)


def hybrid_search(query, top_k=5, alpha=0.5):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
    )[0]

    dense_scores = np.dot(document_embeddings, query_embedding)
    bm25_scores = bm25.get_scores(query.lower().split())

    dense_norm = minmax_normalize(dense_scores)
    bm25_norm = minmax_normalize(bm25_scores)

    result = chunks_df.copy()
    result["dense_score"] = dense_norm
    result["bm25_score"] = bm25_norm
    result["hybrid_score"] = (
        alpha * result["dense_score"]
        + (1 - alpha) * result["bm25_score"]
    )

    return result.sort_values("hybrid_score", ascending=False).head(top_k)


hybrid_results = hybrid_search(
    "Who made Python?",
    top_k=5,
    alpha=0.5,
)

hybrid_results[
    ["title", "text", "dense_score", "bm25_score", "hybrid_score"]
]

,title,text,dense_score,bm25_score,hybrid_score
0,Python,Python is a high-level programming language kn...,1.000000,0.0,0.500000
1,Python,"automation, data science, web development, sci...",0.374260,0.0,0.187130
8,ColBERT,ColBERT is a late-interaction retrieval archit...,0.126825,0.0,0.063412
4,RAG,"Retrieval-Augmented Generation, or RAG, combin...",0.088911,0.0,0.044456
7,SPLADE,SPLADE is a neural sparse retrieval approach. ...,0.075571,0.0,0.037785


## Alternative hybrid fusion: Reciprocal Rank Fusion

Instead of combining raw scores, we can combine rankings.

RRF is commonly written as:

```text
RRF(d) = Σ 1 / (k + rank(d))
```

In [12]:
def reciprocal_rank_fusion(result_lists, k=60, top_k=5):
    scores = {}

    for result in result_lists:
        for rank, row in enumerate(result.itertuples(), start=1):
            doc_id = row.chunk_id
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)

    ranked_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]

    lookup = chunks_df.set_index("chunk_id")
    output = lookup.loc[ranked_ids].reset_index()
    output["rrf_score"] = [scores[x] for x in ranked_ids]

    return output


dense_top = dense_search("Who made Python?", top_k=5)
bm25_top = bm25_search("Who made Python?", top_k=5)

rrf_results = reciprocal_rank_fusion(
    [dense_top, bm25_top],
    top_k=5,
)

rrf_results[["title", "text", "rrf_score"]]

,title,text,rrf_score
0,Python,Python is a high-level programming language kn...,0.032787
1,Python,"automation, data science, web development, sci...",0.032258
2,RAG,"Retrieval-Augmented Generation, or RAG, combin...",0.031010
3,ColBERT,ColBERT is a late-interaction retrieval archit...,0.015873
4,Supervised Learning,Supervised learning trains a machine learning ...,0.015873
